# 06 — RooFit scans of three 4096-event Asimov workspaces

Run this notebook in a **fresh Colab CPU runtime**. It installs ROOT 6.38.00
(with RooFit), compiles `RooSameSampleNormalizer`, reads the three workspaces
from Drive, fits them, and compares their profile likelihoods with
[HEPData Table 2, **Exp NSBI**, four-lepton stat+syst](https://doi.org/10.17182/hepdata.156805.v1/t2).
Notebooks 00–05 are background material; running them is not a prerequisite.

Upload these files, with their existing names, to
**MyDrive/HZZ_OffShell/workspaces/06_root_scans/**:

- `HZZ_4l_true_asimov_4096.root` — true Asimov, 127 constrained NPs.
- `HZZ_4l_data_like_asimov_4096.root` — data-like Asimov, 127 constrained NPs.
- `HZZ_4l_data_like_asimov_4096_pruned54.root` — data-like Asimov, 54 constrained NPs.

All three also profile the three free qqZZ normalizations. The generating
signal strength is 1. The data-like inputs shift the constrained NPs and qqZZ
factors to the observed-data fit, with consistent CR and auxiliary observations.
All three use `RooSameSampleNormalizer` for the interpolated shape at every NP
point, preserving physical rate variations. True versus data-like now isolates
the generating nuisance shift. The pruned model fixes the other 73 alphas at
their generating values. The old anchor-only true file and scan remain separate.
The matching 8192 files are archived for future use, outside this notebook's
default selection; their checksums are in `workspaces_8192.json`.

The full scan has **42 points from 0 to 2**, including 0.01, continuing the
previous study. Allow several hours for all three; Unity timings are not Colab
benchmarks. Each point is saved to Drive. Rerunning resumes completed fits/points
when the workspace, code and ROOT version match. ROOT files are opened read-only;
no `.npy` inputs or precompiled Unity library are needed.


## 1. Install ROOT and get the demonstration code
The pinned [ROOT wheel](https://pypi.org/project/root/0.1a10/) includes ROOT 6.38.00 and RooFit. Its pip packaging is labelled alpha by ROOT. The custom library is rebuilt for this runtime.


In [ ]:
%pip -q install "root==0.1a10" numpy pandas matplotlib

import os, sys, subprocess, json, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
REPO = Path("/content/offshell_reweighting_demo")
BRANCH = "demo/colab-workspace-scans"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/rafaellopesdesa/offshell_reweighting.git", str(REPO)], check=True)
print("Demo revision:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
LIBRARY = REPO / "build/normalizer/libRooSameSampleNormalizer.so"
subprocess.run([sys.executable, str(REPO / "normalizer/build.py"),
                "--output", str(LIBRARY.parent)], check=True)


## 2. Mount Drive and stage the inputs locally
Edit `DRIVE_BASE` if you choose another Drive location. Checksums ensure these are the three preserved workspaces. Only the input staging directory is temporary; results go directly to Drive.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/HZZ_OffShell")
INPUT_DIR = DRIVE_BASE / "workspaces/06_root_scans"
OUTPUT_DIR = DRIVE_BASE / "likelihoods/06_root_scans/full_normalization_v1/root_6_38_00"
LOCAL_INPUTS = Path("/content/offshell_workspace_inputs")
LOCAL_INPUTS.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORKSPACES = json.loads((REPO / "workspaces.json").read_text())

def checksum(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

missing = [s["filename"] for s in WORKSPACES if not (INPUT_DIR / s["filename"]).is_file()]
if missing:
    raise FileNotFoundError(f"Upload to {INPUT_DIR}: {missing}")
for spec in WORKSPACES:
    local = LOCAL_INPUTS / spec["filename"]
    if not local.exists() or checksum(local) != spec["sha256"]:
        shutil.copyfile(INPUT_DIR / spec["filename"], local)
    if checksum(local) != spec["sha256"]:
        raise ValueError(f"Workspace checksum mismatch: {spec['filename']}")
display(pd.DataFrame(WORKSPACES)[["label", "filename", "constrained_nps"]])


## 3. Fit and scan

The compact driver [scan_workspace.py](scan_workspace.py) builds the native
weighted extended RooFit NLL with the ModelConfig constraints and fixed global
observables. Minuit2 finds a continuous global minimum, then fixes **only mu**
at each grid point, reusing Minuit state between neighboring points on each
side of the minimum. No SumW2 correction is applied to these Asimov weights.
The statistic is `2 * (NLL_profile - NLL_global)`; it is not referenced to the
smallest grid value. Fixed/pruned alphas remain fixed.

Every saved point must converge with EDM <= 1e-4, have positive event intensities,
and agree with an independently assembled intensity likelihood. No densities,
process coefficients or negative test statistics are clipped. A failed fit
stops the run. The normalizer dictionary loads before the workspace is read.

For a shorter first demonstration, set `GRID = [0.0, 1.0, 2.0]` or select just
the pruned workspace with `SELECTED = WORKSPACES[-1:]`. Restore the full grid and
rerun to fill the missing points. Each workspace runs in its own process to
release its memory before the next one.


In [ ]:
GRID = sorted(set([0.0, 0.01] + np.round(np.arange(0.05, 2.001, 0.05), 2).tolist()))
SELECTED = WORKSPACES

for spec in SELECTED:
    subprocess.run([sys.executable, "-u", str(REPO / "scan_workspace.py"),
                    "--input", str(LOCAL_INPUTS / spec["filename"]),
                    "--library", str(LIBRARY),
                    "--output", str(OUTPUT_DIR / spec["label"]),
                    "--grid", *map(str, GRID)], check=True)


## 4. Compare with HEPData and save the plot/tables

The checked-in HEPData numbers are unchanged; residuals use only identical mu
coordinates (there is no published point at 0.01). The global-fit point at
(mu_hat, 0) is added only to draw each local curve. The result JSON files also
contain all fitted NP values and diagnostics for every point.


In [ ]:
reference = pd.read_csv(REPO / "reference/hepdata_expected_4l.csv")
fig, (ax, residual) = plt.subplots(2, 1, figsize=(8, 7), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True)
ref = reference[reference.mu.between(min(GRID), max(GRID))]
ax.plot(ref.mu, ref.q, "ko", ms=3, label="HEPData expected NSBI (4l, stat+syst)")
labels = {"true_4096_unpruned": "True Asimov (127 constrained NPs)",
          "data_like_4096_unpruned": "Data-like Asimov (127 constrained NPs)",
          "data_like_4096_pruned54": "Data-like Asimov (54 constrained NPs)"}
summary = []
for spec in SELECTED:
    result = json.loads((OUTPUT_DIR / spec["label"] / "scan.json").read_text())
    best = result["bestfit"]
    table = pd.DataFrame([{k: v for k, v in row.items() if k != "parameters"}
                          for row in result["points"]])
    table = table[table.mu.isin(GRID)].sort_values("mu")
    if set(table.mu) != set(GRID):
        raise RuntimeError(f"Incomplete scan: {spec['label']}")
    table.to_csv(OUTPUT_DIR / f"{spec['label']}_scan.csv", index=False)
    curve = pd.concat([table[["mu", "q"]], pd.DataFrame([{"mu": best["mu_hat"], "q": 0.0}])]).sort_values("mu")
    line, = ax.plot(curve.mu, curve.q, ".-", ms=3, label=labels[spec["label"]])
    shared = table.merge(reference, on="mu", suffixes=("", "_hepdata"))
    shared["delta_q"] = shared.q - shared.q_hepdata
    shared.to_csv(OUTPUT_DIR / f"{spec['label']}_hepdata.csv", index=False)
    residual.plot(shared.mu, shared.delta_q, ".-", color=line.get_color(), ms=3)
    summary.append({"workspace": spec["label"], "mu_hat": best["mu_hat"],
                    "profiled_NPs": result["profiled_nps"], "points": len(table),
                    "RMS_delta_q": np.sqrt(np.mean(shared.delta_q**2)),
                    "global_fit_seconds": best["seconds"],
                    "profile_median_seconds": table.seconds.median(),
                    "process_peak_RSS_GiB": max(best["process_peak_rss_gib"], table.process_peak_rss_gib.max())})
ax.set(ylabel=r"$t_\mu = -2\ln\Lambda(\mu)$", title="4096-event Asimov likelihood scans")
ax.legend(fontsize=9)
residual.axhline(0, color="0.5", lw=1)
residual.set(xlabel=r"Off-shell signal strength $\mu$", ylabel=r"$t_\mu - t_\mu^{\rm HEPData}$", xlim=(min(GRID), max(GRID)))
for axis in (ax, residual):
    axis.grid(alpha=0.2)
for extension in ("png", "pdf"):
    fig.savefig(OUTPUT_DIR / f"asimov_vs_hepdata.{extension}", dpi=180)
plt.show()
summary = pd.DataFrame(summary)
summary.to_csv(OUTPUT_DIR / "fit_summary.csv", index=False)
display(summary)
print("Saved plots, CSV tables and resumable fit JSON to", OUTPUT_DIR)
